# 16S Analyses of the Longitudinal Acne Study
## Relative Abundance Plots

Date created: 10/15/2024

Notebook authors: Yang Chen and Britta De Pessemier

Data analysis by: Tyler Myers, Britta De Pessemier, and Yang Chen

This notebook plots the following:

- 16S V1-V3 and V4 relative abundance plots at Genus taxon level aggregated by skin group
- 16S V1-V3 and V4 relative abundance plots at Genus taxon level by each sample

In [195]:
# Import Python packages
import pandas as pd
import numpy as np
import biom
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import cycle
import os
from matplotlib.colors import ListedColormap
from matplotlib.colors import to_rgba


In [196]:
# Load the metadata
metadata_path = '../Metadata/metadata_final_22102024.tsv'
metadata = pd.read_csv(metadata_path, sep='\t')
metadata['severity_group'].value_counts()

severity_group
low Acne_L         70
moderate Acne_L    64
absent Healthy     57
absent Acne_NL     27
high Acne_L        25
low Acne_NL        23
Name: count, dtype: int64

In [197]:
metadata = pd.read_csv(metadata_path, sep="\t")

metadata['SampleID'] = metadata['SampleID'].astype(str)
metadata = metadata.set_index('SampleID')
metadata


,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,taxon_id,...,a,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group,severity_level,severity_group
SampleID,,,,,,,,,,,,,,,,,,,,,
LAMI.RD308.D16.C1,C1,not applicable,Lesional,skin,Day 16,not applicable,16,308,not applicable,539655,...,33.765638,acne,RD308,acne,PP_308,PP_308C1,Lesional_C1,Acne_L,moderate,moderate Acne_L
LAMI.RD310.D21.C1,C1,72,Lesional,skin,Day 21,36,21,310,17,539655,...,31.919478,acne,RD310,acne,PP_310,PP_310C1,Lesional_C1,Acne_L,moderate,moderate Acne_L
LAMI.RD305.D21.C3,C3,69,Non-lesional,skin,Day 21,26,21,305,25,539655,...,22.152503,acne,RD305,healthy,PP_305,PP_305C3,Non-lesional_C3,Acne_NL,absent,absent Acne_NL
LAMI.RD306.D18.C2,C2,not applicable,Lesional,skin,Day 18,not applicable,18,306,not applicable,539655,...,27.397918,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L,low,low Acne_L
LAMI.RD306.D7.C2,C2,90,Lesional,skin,Day 7,13,7,306,23,539655,...,28.819451,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L,moderate,moderate Acne_L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD317.D21.C1,C1,77,Lesional,skin,Day 21,19,21,317,20,539655,...,21.946648,acne,RD317,acne,PP_317,PP_317C1,Lesional_C1,Acne_L,low,low Acne_L
LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,539655,...,26.344240,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy
LAMI.RD014.D14.C2,C2,not applicable,Non-lesional,skin,Day 14,not applicable,14,14,not applicable,539655,...,16.359699,control,RD014,healthy,PP_14,PP_14C2,Non-lesional_C2,Healthy,absent,absent Healthy


In [198]:
# Define paths to the collapsed taxa tables
biom_paths = {
    'V1-V3': '../Data/16S/Tables/179426_feature-table_16S_V1V3_rare-11054_sampleIDfixed_Genus-ASV_SILVA.biom',
    'V4': '../Data/16S/Tables/174951_feature-table_16S_V4_rare-3769_sampleIDfixed_Genus-ASV_SILVA.biom'
}

In [199]:
# Predefined color palette for specific taxon
taxa_colors = {
    'g__Cutibacterium ASV-1': '#ffa505',  # Bright orange
    'g__uncultured ASV-1': '#585858',         # Dark grey
    'g__Staphylococcus ASV-1': '#92f0f0',      # Fluorescent light blue
    'g__Streptococcus ASV-1': '#e2b46c',    # Beige
    'g__Corynebacterium ASV-1': '#ffe59a',        # Pastel yellow
    'g__Lawsonella ASV-1': '#70a8dc',         # Light blue
    'g__Veillonella ASV-1': '#c5bce0',         # Pastel purplish
    'g__Micrococcus ASV-1':'#f4cccd',           # Pastel yellow
    'g__Alloprevotella ASV-1': '#8c5079',        # Plum
    'g__Lactobacillus ASV-1': '#daead3',     # Pale mint green
    'g__Neisseria ASV-1': '#f6475f',         # Redish pink
    'g__Anaerococcus ASV-1': '#008000',         # Green
    'Others': '#ededed'                 # White
}

In [200]:
# A list of unique colors to use for taxa not predefined
unique_colors = sns.color_palette("deep", n_colors=20).as_hex()
unique_color_iter = cycle(unique_colors)  # Iterator to cycle through unique colors

In [201]:
# Function to load BIOM table, collapse by taxa, sort rows by row sum, remove specified samples, and convert to relative abundance
def load_biom_table(biom_path, metadata_path):
    # Load metadata as a DataFrame from the file path
    metadata = pd.read_csv(metadata_path, sep='\t')

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)
    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    # Replace 'uncultured' row with 'uncultured Neisseriaceae'
    # df = df.rename(index={' g__uncultured': ' g__uncultured Neisseriaceae'})
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])
    
    # Convert the table to relative abundances
    df = df.div(df.sum(axis=0), axis=1)
    
    return df


In [202]:
# Function to determine the top n taxon and collapse the rest as "Others"
def collapse_top_n(df):
    top_taxa = df.sum(axis=1).nlargest(20).index  # Select top n taxon
    df_top = df.loc[top_taxa]
    df_top.loc['Others'] = df.loc[~df.index.isin(top_taxa)].sum()
    return df_top

In [203]:
# Function to get or assign colors to taxon
def get_taxa_colors(taxon, global_taxa_color_map):
    for taxa in taxon:
        if taxa not in global_taxa_color_map:
            if taxa in taxa_colors:
                global_taxa_color_map[taxa] = taxa_colors[taxa]
            else:
                global_taxa_color_map[taxa] = next(unique_color_iter)  # Assign a new unique color
    return global_taxa_color_map

## Relative abundance plots

In [204]:
def plot_relative_abundance(df, metadata, group_column, output_dir, key, taxa_color_map):

    # Align samples between table and metadata
    shared_samples = df.columns.intersection(metadata.index)
    if len(shared_samples) == 0:
        raise ValueError("No overlapping samples between table and metadata.")

    df = df.loc[:, shared_samples]
    metadata = metadata.loc[shared_samples]

    # Average by group
    df_grouped = df.groupby(metadata[group_column], axis=1).mean()

    desired_order = ['Healthy', 'Acne_NL', 'Acne_L']
    present_groups = [g for g in desired_order if g in df_grouped.columns]
    df_grouped = df_grouped[present_groups]

    # Compute group sizes dynamically
    group_sizes = (
        metadata[group_column]
        .value_counts()
        .reindex(present_groups)
    )

    # Output file path
    os.makedirs(output_dir, exist_ok=True)
    output_png_file = os.path.join(
        output_dir, f'Relative_abundance_{key}_Genus-ASV.png'
    )

    # Plot title
    if key == 'V1-V3':
        plot_title = '16S rRNA (V1–V3) Relative Abundance'
    elif key == 'V4':
        plot_title = '16S rRNA (V4) Relative Abundance'
    else:
        plot_title = 'Relative Abundance'

    # Plot stacked bars
    ax = df_grouped.T.plot(
        kind='bar',
        stacked=True,
        figsize=(8.5, 10),
        width=0.8,
        color=[taxa_color_map.get(t, '#ADD8E6') for t in df_grouped.index]
    )

    plt.ylabel('Relative Abundance', fontsize=18)
    plt.xlabel(' ')
    plt.title(plot_title, fontsize=20)

    # Dynamic x-axis labels
    label_map = {
        'Healthy': 'Healthy',
        'Acne_NL': 'Acne NL',
        'Acne_L': 'Acne L'
    }

    new_labels = [
        f"{label_map[g]}\n(n={group_sizes[g]})"
        for g in present_groups
    ]

    plt.xticks(
        ticks=range(len(new_labels)),
        labels=new_labels,
        rotation=0,
        ha='center',
        fontsize=18
    )
    plt.yticks(fontsize=18)

    plt.legend(
        loc='center left',
        bbox_to_anchor=(1.0, 0.5),
        fontsize=16,
        title='Genus',
        title_fontsize=18
    )

    plt.tight_layout()
    plt.savefig(output_png_file, dpi=600)
    plt.close()


In [205]:
# Load metadata once (indexed by SampleID)
metadata_idx = metadata if metadata.index.name == 'SampleID' else metadata.set_index('SampleID')

# Get sample sets for each BIOM
biom_sample_sets = {}

for key, biom_path in biom_paths.items():
    df_tmp = load_biom_table(biom_path, metadata_path)
    biom_sample_sets[key] = set(df_tmp.columns)

# Intersection across all BIOMs
shared_samples_all = set.intersection(*biom_sample_sets.values())

# Also enforce presence in metadata
shared_samples_all = shared_samples_all.intersection(metadata_idx.index)

shared_samples_all = sorted(shared_samples_all)

print("Shared samples across all BIOMs:", len(shared_samples_all))


Shared samples across all BIOMs: 184


In [206]:
global_taxa_color_map = {}

for key, biom_path in biom_paths.items():

    df = load_biom_table(biom_path, metadata_path)

    # Enforce identical samples across BIOMs
    df = df.loc[:, shared_samples_all]
    metadata_subset = metadata_idx.loc[shared_samples_all]

    df_top_15 = collapse_top_n(df)

    output_dir = '../Figures/Main/Figure_3/'
    os.makedirs(output_dir, exist_ok=True)

    print(f"{key}: using {df.shape[1]} aligned samples")

    global_taxa_color_map = get_taxa_colors(
        df_top_15.index,
        global_taxa_color_map
    )

    plot_relative_abundance(
        df_top_15,
        metadata_subset,
        'group',
        output_dir,
        key,
        global_taxa_color_map
    )


V1-V3: using 184 aligned samples


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_48236/2681546954.py:12: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  df_grouped = df.groupby(metadata[group_column], axis=1).mean()


V4: using 184 aligned samples


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_48236/2681546954.py:12: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  df_grouped = df.groupby(metadata[group_column], axis=1).mean()
